In [ ]:
import pandas as pd
import numpy as np

In [ ]:
domainarch = pd.read_csv("../data/promiscuity/domains_per_entry.tsv",sep="\t")
promiscuity = pd.read_csv("../data/promiscuity/converted_protein_domains_with_promiscuity_flag.tsv", sep="\t", usecols= ['Entry', 'is_promiscuous'])
dom_prom = domainarch.merge(promiscuity, on='Entry', how='left')
dom_prom

In [ ]:
annot = pd.read_csv('../data/annotation/fastaid_annot_id_clusts_473het_annotperdomain_tocheck.csv',usecols = ['Entry', 'Organism','Organism (ID)','fastaID','sequence', 'GH18sequence','signature_accession'])
dom_prom_org = dom_prom.merge(annot, on='Entry', how='left')
dom_prom_org

CID annotations

In [ ]:
!hmmsearch --tblout ../data/annotation/CID_Library_hmmsearch_allseq_251121_1and0.tbl --cpu 5 ../data/annotation/251115_CIDLibrary_1and0.hmm ../data/annotation/GH18_domains.fasta


In [ ]:
import pandas as pd

hmmsearch_path = "../data/annotation/2260926_CID_Library_hmmsearch_allseq_1and0.tbl"
out_path = "../data/annotation/260926_CID_Library_hmmsearch_allseq_1and0.csv"

cols = ["target_name", "accession", "query_name", "accession.1",
        "E-value", "score", "bias",            # full sequence
        "E-value.1", "score.1", "bias.1",      # best 1 domain
        "exp", "reg", "clu", "ov", "env", "dom", "rep", "inc",
        "description"]

rows = []
with open(hmmsearch_path) as fh:
    for line in fh:
        if line.startswith("#") or not line.strip():
            continue
        rows.append(line.rstrip("\n").split(maxsplit=18))

df = pd.DataFrame(rows, columns=cols)
df[cols[4:18]] = df[cols[4:18]].apply(pd.to_numeric)

df = df.sort_values(["score", "E-value"], ascending=[False, True], kind="stable")
df = df.drop_duplicates(subset="target_name", keep="first")
df.to_csv(out_path, index=False)
print(len(rows), "hits parsed ->", len(df), "unique targets")

In [ ]:
indelclass = pd.read_csv("../data/annotation/260926_CID_Library_hmmsearch_allseq_1and0.csv", usecols=['target_name','query_name','score'])
#indelclass['Entry'] = indelclass['target_name'].str.split("|").str[0]
dom_prom_org_cid = dom_prom_org.merge(indelclass, right_on='target_name',left_on='fastaID', how='left')
#organise by score
dom_prom_org_cid = dom_prom_org_cid.sort_values(["fastaID", "score"], ascending=[True, False], kind="stable")
dom_prom_org_cid = dom_prom_org_cid.drop_duplicates(subset=['fastaID'], keep='first')
dom_prom_org_cid

In [ ]:
cat = pd.read_csv('../data/annotation/05_master_categories/signature_broad_categories_global_mapping.csv')
cat

In [ ]:
# Accession -> sorted list of unique categories
sig_to_cats = (
    cat[['signature_accession', 'broad_category']]
    .dropna()
    .groupby('signature_accession')['broad_category']
    .agg(lambda s: sorted(set(s)))
    .to_dict()
)

def domains_to_categories(dom_str):
    if pd.isna(dom_str):
        return np.nan
    slots = []
    for token in str(dom_str).split():
        cats = sig_to_cats.get(token)
        if cats:
            slots.append("/".join(cats))
    return "|".join(slots) if slots else np.nan

dom_prom_org_cid['broad_categories'] = dom_prom_org_cid['domains'].apply(domains_to_categories)
dom_prom_org_cid

In [ ]:
bc = dom_prom_org_cid['broad_categories'].dropna()

# 1. More than one category slot (multiple categorised domains, even if they share a category)
multi_slot = bc.str.split('|').str.len() > 1

# 2. More than one *distinct* category (counting alternatives within ambiguous domains)
n_distinct = bc.str.split(r'[|/]', regex=True).apply(lambda x: len(set(x)))
multi_distinct = n_distinct > 1

print(f"Rows with >1 category slot:     {multi_slot.sum()}")
print(f"Rows with >1 distinct category: {multi_distinct.sum()}")
print(f"Out of {len(bc)} categorised rows ({len(dom_prom_org_cid)} total)")

In [ ]:
import pandas as pd
from pathlib import Path

# --- Input dataframe ---
# df_main MUST contain column "target_name"
# Example: df_main = pd.read_csv("yourfile.tsv", sep="\t")

# Extract IDs before the "|"
df = pd.DataFrame({
    "id": dom_prom_org_cid["fastaID"]
})

# --- Config directories ---
primary_dir    = Path("../data/annotation/05_master_categories/primary_clusters")
secondary_dir  = Path("../data/annotation/05_master_categories/secondary_clusters")
#out_path       = Path("treeseq_with_clusters.tsv")

# --- Helper to load IDs from cluster files ---
def load_id_set(path: Path):
    if not path.is_file():
        print(f"WARNING: file not found: {path}")
        return set()
    with path.open() as f:
        return {line.strip() for line in f if line.strip()}

# --- Primary clusters (higher priority) ---
primary_files = {
    primary_dir / "gh18domIPscan_cluster1_bitscore96_accessions.txt":  "cluster_3",
    primary_dir / "gh18domIPscan_cluster2_bitscore96_accessions.txt":  "cluster_2",
    primary_dir / "gh18domIPscan_cluster1_bitscore162_accessions.txt": "cluster_1.1",
    primary_dir / "gh18domIPscan_cluster0_bitscore162_accessions.txt": "cluster_1.2",
}

# --- Secondary clusters (lower priority) ---
secondary_files = {
    secondary_dir / "treeseq_cluster0_bitscore163_accessions.txt": "cluster_1.2",
    secondary_dir / "treeseq_cluster1_bitscore163_accessions.txt": "cluster_1.1",
    secondary_dir / "treeseq_cluster1_bitscore97_accessions.txt":  "cluster_3",
    secondary_dir / "treeseq_cluster2_bitscore97_accessions.txt":  "cluster_2",
}

# Load sets in priority order
primary_sets = [(label, load_id_set(path)) for path, label in primary_files.items()]
secondary_sets = [(label, load_id_set(path)) for path, label in secondary_files.items()]

# --- Assign clusters ---
def assign_cluster(seq_id: str) -> str:
    # Primary clusters first
    for label, idset in primary_sets:
        if seq_id in idset:
            return label
    # Secondary clusters
    for label, idset in secondary_sets:
        if seq_id in idset:
            return label
    # No match
    return "unassigned"

df["cluster"] = df["id"].apply(assign_cluster)

# --- Save + view ---
#df.to_csv(out_path, sep="\t", index=False)
df

#print(f"\nSaved with clusters to: {out_path}")
print(df["cluster"].value_counts(dropna=False))


In [ ]:
dom_prom_org_cid_ssn = dom_prom_org_cid.merge(df, left_on='fastaID', right_on='id',how='left')
dom_prom_org_cid_ssn = dom_prom_org_cid_ssn.drop_duplicates(subset=['fastaID'])
dom_prom_org_cid_ssn

In [ ]:
#print all the diff values in broad_categories and columns and how many of each
for col in dom_prom_org_cid_ssn.columns:
    print(f"\nColumn: {col}")
    print(dom_prom_org_cid_ssn[col].value_counts(dropna=False))   


In [ ]:
import pandas as pd
from pathlib import Path

# dataframe you already have
df = dom_prom_org_cid_ssn.copy()

# ensures Entry is string
df["fastaID"] = df["fastaID"].astype(str)

# directory containing the clade files
clade_dir = Path(".")

# mapping: clade file → label
clade_files = {
    "../data/annotation/05_master_categories/clade1.txt": "clade1",
    "../data/annotation/05_master_categories/clade2.txt": "clade2",
    "../data/annotation/05_master_categories/clade3.txt": "clade3",
    "../data/annotation/05_master_categories/clade4.txt": "clade4",
    "../data/annotation/05_master_categories/clade5.txt": "clade5",
}

# load each clade file into a dict of sets
clade_sets = {}
for filename, label in clade_files.items():
    path = clade_dir / filename
    if not path.is_file():
        print(f"WARNING: missing {filename}")
        clade_sets[label] = set()
        continue
    
    with path.open() as f:
        ids = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            # split on '|' and take first part
            ids.append(line.lstrip(">").removesuffix("|ref"))
        
        clade_sets[label] = set(ids)

# assignment function
def assign_clade(entry):
    for label, idset in clade_sets.items():
        if entry in idset:
            return label
    return "unassigned"

# apply
df["clade"] = df["fastaID"].apply(assign_clade)

df#.head()


In [ ]:
# lookup: accession -> fastaID(s) in df
acc_to_df = (df["fastaID"].astype(str)
               .groupby(df["fastaID"].astype(str).str.split("|").str[0])
               .apply(set).to_dict())
df_ids = set(df["fastaID"].astype(str))

for label, idset in clade_sets.items():
    fixed = set()
    for i in idset:
        if i in df_ids:
            fixed.add(i)                      # already matches, leave alone
            continue
        acc = i.split("|")[0]
        hits = acc_to_df.get(acc, set())
        if len(hits) == 1:
            new = next(iter(hits))
            print(f"{label}: {i} -> {new}")
            fixed.add(new)
        elif len(hits) > 1:
            print(f"{label}: {i} AMBIGUOUS -> assigning to all {sorted(hits)}")
            fixed.update(hits)
        else:
            print(f"{label}: {i} NOT FOUND even by accession")
            fixed.add(i)
    clade_sets[label] = fixed

In [ ]:
#print all the diff values in broad_categories and columns and how many of each
for col in df.columns:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False))   

In [ ]:
df_ids = set(df["fastaID"].astype(str))

for label, idset in clade_sets.items():
    missing = sorted(idset - df_ids)
    print(f"{label}: {len(missing)}/{len(idset)} IDs not found in dataframe")
    for i in missing:
        print("   ", i)

In [ ]:
print(df["clade"].value_counts(dropna=False))


In [ ]:
tok = df["domains"].fillna("").astype(str).str.split()

df["n_domains"] = tok.str.len()
df["n_GH18_domains"] = tok.apply(lambda t: t.count("GH18"))   # exact token match only

print(df[["fastaID", "domains", "n_domains", "n_GH18_domains"]].head())
dftoadd = df.copy()

## fetech taxonomic info or load in existing file

In [ ]:
# fetch taxonomic info and create final database
from Bio import Seq
from Bio import SeqIO
from Bio import Entrez
import pandas as pd
from tqdm import tqdm
import seaborn as sns
from datetime import datetime

# set the email address for Entrez
Entrez.email = "your.email@example.com"

# define a list of taxids
#taxid_list = ['2250085',

#only include unique taxids
# define a list of taxids
taxid_list = df['Organism (ID)'].to_list()
#only include unique taxids
taxid_list = list(set(taxid_list))

#accesions_list = genpept_df['Entry'].to_list()

# define a batch size for fetching records from Entrez
batch_size = 500

# loop over the taxids in batches and retrieve their lineage information
taxonomy_data = []
for i in range(0, len(taxid_list), batch_size):
    taxid_batch = taxid_list[i:i+batch_size]
    handle = Entrez.efetch(db="taxonomy", id=taxid_batch, retmode="xml")
    records = Entrez.read(handle, validate=False)
    for record in records:
        try:
            taxonomy_dict = {"TaxID": record["TaxId"]}
            lineage = record["LineageEx"]
            for data in lineage:
                rank = data["Rank"]
                name = data["ScientificName"]
                taxonomy_dict[rank] = name
            taxonomy_data.append(taxonomy_dict)
            print(record["TaxId"])
        except KeyError:
            print(f"No lineage information found for TaxID {record['TaxId']}")

# Create a DataFrame from the taxonomic information
taxa_df2 = pd.DataFrame(taxonomy_data)

#taxa_df2 = taxa_df2[['TaxID', 'superkingdom','kingdom','phylum', 'class', 'order', 'family', 'genus', 'species']]
#taxa_df2 = taxa_df2[['TaxID', 'superkingdom','kingdom','phylum']]

desired_cols = ['TaxID','superkingdom','kingdom','phylum','class','order','family','genus','species']
existing_cols = [c for c in desired_cols if c in taxa_df2.columns]
taxa_df2 = taxa_df2[existing_cols]


taxa_df2['TaxID'] = taxa_df2['TaxID'].astype(str)



In [ ]:
taxa_df2 = taxa_df2.astype(str)
df_taxa = df.merge(taxa_df2, left_on = 'Organism (ID)', right_on = 'TaxID', how='left')
df_taxa

In [ ]:
taxa = pd.read_csv('../data/annotation/singleseq_perline_masterannot_allseq_taxasp_39582_260805_260925.csv',usecols=['Entry','kingdom', 'phylum', 'class', 'order',
       'family', 'genus'])
df_taxa = dftoadd.merge(taxa, on = 'Entry',  how='left')
df_taxa = df_taxa.drop_duplicates(subset='fastaID')
df_taxa

In [ ]:
taxa['phylum'].isna().sum()


In [ ]:
sp = pd.read_csv('../data/annotation/05_master_categories/signalp_parsed_with_details.csv',usecols = ['id','Prediction'])
sp

In [ ]:

df_taxa_sp = df_taxa.merge(sp, left_on='Entry', right_on = 'id', how='left')
df_taxa_sp

In [ ]:
df_taxa_sp.to_csv('../data/annotation/260926_perdomain_annotations.csv')

In [ ]:
df_singleseq.columns

In [ ]:
df_singleseq = df_taxa_sp.drop_duplicates(subset='Entry')
df_singleseq = df_singleseq.drop(columns=['signature_accession', 'GH18sequence', 'fastaID','target_name', 'query_name', 'score','id_x', 'cluster', 'clade','id_y'])
df_singleseq.to_csv('../data/annotation/260926_perseq_annotations.csv')

In [ ]:
df_assigned = df_taxa_sp[df_taxa_sp["clade"] != "unassigned"].copy()
df_assigned

# Figure 4: Source Data (proportion tables)
1. **Phylum × InDel class:** full dataset, one count per protein per class (as in Supplementary Figure 9).
2. **Broad category × InDel class:** full dataset, auxiliary-domain annotations (as in Supplementary Figure 10).
3. **Signal peptide × phylogenetic group:** all tree tips listed in `clade1.txt`–`clade5.txt` (UniProt and NCBI), one count per protein.

Each table is written as proportions, with a matching counts table so every value can be recomputed.

In [ ]:
# Cell 1 — inputs
import os, pandas as pd

INFILE      = "../data/annotation/260926_perdomain_annotations.csv"
OUT_DIR     = "../results/source_data"
CID2CLASS   = {"CID2": "A", "CID1and0": "B", "CID3": "C", "CID4": "D", "CID5": "E", "CID6": "F", "CID7": "G"}
CLASSES     = list("ABCDEFG")
os.makedirs(OUT_DIR, exist_ok=True)

d = pd.read_csv(INFILE, low_memory=False, usecols=["Entry", "query_name", "phylum", "broad_categories"])
d["InDel_class"] = d.query_name.map(CID2CLASS)
DROP_UNASSIGNED_PHYLUM = True   # True: leave out proteins with no phylum (NaN) from the phylum table

d["phylum"] = d.phylum.fillna("Unassigned")
units = d.dropna(subset=["InDel_class"]).drop_duplicates(["Entry", "InDel_class"])   # one protein per class

def write(counts, name, index_name):
    counts = counts.rename_axis(index=index_name, columns=None)
    prop = counts / counts.sum()
    counts.to_csv(f"{OUT_DIR}/{name}_counts.csv", encoding="utf-8-sig")
    prop.to_csv(f"{OUT_DIR}/{name}_proportions.csv", encoding="utf-8-sig")
    return prop

In [ ]:
# Cell 2 — phylum x InDel class (full dataset)
_ph = units[units.phylum != "Unassigned"] if DROP_UNASSIGNED_PHYLUM else units
tab_phylum = pd.crosstab(_ph.phylum, _ph.InDel_class).reindex(columns=CLASSES, fill_value=0)
fig4_phylum = write(tab_phylum, "Fig4_phylum_by_InDel_class", "phylum")
fig4_phylum.loc[fig4_phylum.max(axis=1).sort_values(ascending=False).index[:12]].round(3)

In [ ]:
# Cell 3 — broad category x InDel class (full dataset)
ann = (units.assign(broad_category=units.broad_categories.str.split("|"))
            .explode("broad_category").dropna(subset=["broad_category"]).reset_index(drop=True))
tab_category = pd.crosstab(ann.broad_category, ann.InDel_class).reindex(columns=CLASSES, fill_value=0)
fig4_category = write(tab_category, "Fig4_broad_category_by_InDel_class", "broad_category")
fig4_category.round(3)

In [ ]:
# Cell 4 — signal peptide x phylogenetic group (tips listed in clade1–5.txt)
CLADE_DIR   = "../data/annotation/05_master_categories"            # clade1.txt … clade5.txt (one tip ID per line)
SIGNALP_CSV = "../data/annotation/05_master_categories/signalp_parsed_with_details.csv"
GROUP_OF    = {"clade1": "I", "clade2": "II", "clade3": "III", "clade4": "IV", "clade5": "V"}
SP_LABEL    = {"SP": "Sec/SPI", "LIPO": "Sec/SPII", "TAT": "Tat/SPI", "TATLIPO": "Tat/SPII", "OTHER": "No signal peptide"}

tips = pd.concat([pd.DataFrame({"tip": [l.strip() for l in open(f"{CLADE_DIR}/{c}.txt") if l.strip()], "clade": c})
                  for c in GROUP_OF], ignore_index=True)
assert not tips.tip.duplicated().any(), "a tip is listed in more than one clade file"
tips["accession"] = tips.tip.str.split("|").str[0]                 # UniProt 'A0A4R5C255|1of1' or NCBI 'MBJ13210.1'
tips["group"] = tips.clade.map(GROUP_OF)

sp = pd.read_csv(SIGNALP_CSV, usecols=["id", "Prediction"]).drop_duplicates("id")
tips = tips.merge(sp, left_on="accession", right_on="id", how="left")
missing = tips.Prediction.isna().sum()
assert missing == 0, f"{missing} tips have no SignalP prediction"
tips["signal_peptide"] = tips.Prediction.map(SP_LABEL)

# a protein with two GH18 tips in the same clade (…|1of2, …|2of2) is counted once
seqs = tips.drop_duplicates(["accession", "group"])
print(f"{len(tips)} tips -> {len(seqs)} proteins "
      f"({len(tips) - len(seqs)} second-domain tips of the same protein collapsed)")

tab_sp = pd.crosstab(seqs.signal_peptide, seqs.group).reindex(columns=["I", "II", "III", "IV", "V"], fill_value=0)
fig4_sp = write(tab_sp, "Fig4_signal_peptide_by_group", "signal_peptide")
seqs[["tip", "accession", "clade", "group", "Prediction", "signal_peptide"]].to_csv(
    f"{OUT_DIR}/Fig4_signal_peptide_by_group_per_protein.csv", index=False, encoding="utf-8-sig")
display(tab_sp)
fig4_sp.round(3)

## Figure 3 Signal Peptide insets

In [ ]:
# ==== Figure 3c–d — SignalP 6.0 predictions per phylogenetic group + pairwise Cramér's V (with Source Data) ====
import itertools, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import chi2_contingency
from matplotlib.colors import LinearSegmentedColormap

CLADE_DIR   = "../data/annotation/05_master_categories"            # clade1.txt … clade5.txt
SIGNALP_CSV = "../data/annotation/05_master_categories/signalp_parsed_with_details.csv"
FIG_DIR     = "../results/figures"; SRC_DIR = "../results/source_data"
UNIT        = "protein"      # "protein": both GH18 tips of one protein in the same clade count once; "tip": every tip counts
GROUP_OF    = {"clade1": "I", "clade2": "II", "clade3": "III", "clade4": "IV", "clade5": "V"}
GROUPS      = list(GROUP_OF.values())
PRED_ORDER  = ["OTHER", "TAT", "LIPO", "SP"]                       # plotting order (as in the original Figure 3c)
PRED_COLORS = {"OTHER": "#d9daea", "TAT": "#9d9bc7", "LIPO": "#736db0", "SP": "#353165", "TATLIPO": "#b9b7d9"}
PRED_LABEL  = {"OTHER": "No signal peptide", "TAT": "Tat/SPI", "LIPO": "Sec/SPII", "SP": "Sec/SPI", "TATLIPO": "Tat/SPII"}
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(SRC_DIR, exist_ok=True)

# ---- tips -> group -> SignalP prediction ----
tips = pd.concat([pd.DataFrame({"tip": [l.strip() for l in open(f"{CLADE_DIR}/{c}.txt") if l.strip()], "clade": c})
                  for c in GROUP_OF], ignore_index=True)
assert not tips.tip.duplicated().any(), "a tip is listed in more than one clade file"
tips["accession"] = tips.tip.str.split("|").str[0]                 # UniProt 'A0A4R5C255|1of1' or NCBI 'MBJ13210.1'
tips["group"] = tips.clade.map(GROUP_OF)
sp = pd.read_csv(SIGNALP_CSV, usecols=["id", "Prediction"]).drop_duplicates("id")
tips = tips.merge(sp, left_on="accession", right_on="id", how="left").drop(columns="id")
assert tips.Prediction.notna().all(), f"{tips.Prediction.isna().sum()} tips have no SignalP prediction"
units = tips.drop_duplicates(["accession", "group"]) if UNIT == "protein" else tips
print(f"{len(tips)} tips -> {len(units)} {UNIT}s")

preds = PRED_ORDER + [p for p in sorted(units.Prediction.unique()) if p not in PRED_ORDER]   # keep e.g. TATLIPO if present
counts = pd.crosstab(units.Prediction, units.group).reindex(index=preds, columns=GROUPS, fill_value=0)
props  = counts / counts.sum()

# ---- Figure 3c: proportion of SignalP classes per group ----
long = props.rename_axis("prediction").reset_index().melt(id_vars="prediction", var_name="group", value_name="prop")
plt.figure(figsize=(10, 6))
sns.barplot(data=long, x="group", y="prop", hue="prediction", hue_order=preds, order=GROUPS,
            palette={p: PRED_COLORS.get(p, "#999999") for p in preds})
plt.ylabel("Proportion"); plt.xlabel("Group")
handles, _ = plt.gca().get_legend_handles_labels()
plt.legend(handles, [PRED_LABEL.get(p, p) for p in preds], bbox_to_anchor=(1.05, 1), loc="upper left",
           title="SignalP 6.0 prediction")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/Fig3c_signalp_by_group.png", dpi=600); plt.savefig(f"{FIG_DIR}/Fig3c_signalp_by_group.pdf")
plt.show()

# ---- Figure 3d: pairwise Cramér's V between groups (2 x k chi-square, no continuity correction) ----
rows = []
for g1, g2 in itertools.combinations(GROUPS, 2):
    t = counts[[g1, g2]].T
    t = t.loc[:, t.sum() > 0]                                          # drop classes absent from both groups
    chi2, p, dof, _ = chi2_contingency(t.values, correction=False)
    n = t.values.sum()
    rows.append({"group_1": g1, "group_2": g2, "n_1": int(t.loc[g1].sum()), "n_2": int(t.loc[g2].sum()),
                 "chi2": chi2, "dof": dof, "p": p, "cramers_V": np.sqrt(chi2 / (n * (min(t.shape) - 1)))})
cv = pd.DataFrame(rows)
V_mat = pd.DataFrame(0.0, index=GROUPS, columns=GROUPS)
for r in cv.itertuples():
    V_mat.loc[r.group_1, r.group_2] = V_mat.loc[r.group_2, r.group_1] = r.cramers_V

plt.figure(figsize=(6, 5))
sns.heatmap(V_mat, vmin=0, vmax=1, cmap=LinearSegmentedColormap.from_list("lavender_indigo", ["#ffffff", "#353165"]),
            annot=True, fmt=".2f", cbar_kws={"label": "Cramér's V"})
plt.ylabel("Group"); plt.xlabel("Group")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/Fig3d_signalp_cramersV.png", dpi=600); plt.savefig(f"{FIG_DIR}/Fig3d_signalp_cramersV.pdf")
plt.show()

# ---- Source Data ----
units.assign(signal_peptide=units.Prediction.map(PRED_LABEL))[["tip", "accession", "clade", "group", "Prediction", "signal_peptide"]] \
     .to_csv(f"{SRC_DIR}/Fig3cd_signalp_per_{UNIT}.csv", index=False, encoding="utf-8-sig")
counts.rename(index=PRED_LABEL).rename_axis("signal_peptide").to_csv(f"{SRC_DIR}/Fig3c_signalp_counts.csv", encoding="utf-8-sig")
props.rename(index=PRED_LABEL).rename_axis("signal_peptide").to_csv(f"{SRC_DIR}/Fig3c_signalp_proportions.csv", encoding="utf-8-sig")
cv.to_csv(f"{SRC_DIR}/Fig3d_signalp_pairwise_cramersV.csv", index=False, encoding="utf-8-sig")

# round trip: plotted values recomputable from the per-unit file
chk = pd.read_csv(f"{SRC_DIR}/Fig3cd_signalp_per_{UNIT}.csv", encoding="utf-8-sig", keep_default_na=False)
assert pd.crosstab(chk.Prediction, chk.group).reindex(index=preds, columns=GROUPS, fill_value=0).equals(counts)
display(counts.rename(index=PRED_LABEL)); display(props.rename(index=PRED_LABEL).round(3))
display(cv[["group_1", "group_2", "n_1", "n_2", "chi2", "p", "cramers_V"]].round(3))